# PINN for Composite Interface Parameter Identification - Quick Start

This notebook demonstrates how to use the trained PINN model to predict Extended Interface parameters from 3-Layer Interphase effective properties.

## Workflow
1. Define material properties for a composite
2. Compute target K_eff, G_eff from 3-Layer Interphase model
3. Use PINN to predict interface parameters (k_bar, lambda_bar, mu_bar, alpha)
4. Verify with Extended Interface model

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np

from src.models.pinn import PINN
from src.physics.extended_interface import compute_effective_properties
from src.physics.three_layer_interphase import compute_3layer_properties

print("Imports successful!")

## Load the Pre-trained PINN Model

In [ ]:
# Load V2 model (best performance - 100% pass rate)
model_path = project_root / 'checkpoints' / 'v2' / 'pinn_best.pt'

checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
model = PINN(hidden_dims=(256, 256, 256), norm_params=checkpoint.get('norm_params'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded from: {model_path}")
print(f"Training epoch: {checkpoint.get('epoch', 'N/A')}")

## Define Material Properties

We define a composite with:
- **Soft particles** (stiffness ratio SR = 0.1)
- **Volume fraction** f = 0.3
- **3-Layer interphase** with phi ratios [0.2, 0.3, 0.4]

In [ ]:
# Material properties
kappa_mat = 17.33     # Matrix bulk modulus
mu_mat = 8.0          # Matrix shear modulus

# Soft particles (SR = 0.1)
SR = 0.1
kappa_inc = kappa_mat * SR  # = 1.73
mu_inc = mu_mat * SR        # = 0.8

# Geometry
f = 0.3               # Volume fraction
phi_1, phi_2, phi_3 = 0.2, 0.3, 0.4  # Interphase layer ratios

# Size parameter
sz = 0.00278
R = ((3 * f) / (4 * np.pi)) ** (1/3) * sz  # Particle radius

print("Material Properties:")
print(f"  Matrix: kappa={kappa_mat}, mu={mu_mat}")
print(f"  Particle: kappa={kappa_inc:.2f}, mu={mu_inc:.2f} (SR={SR})")
print(f"  Volume fraction: f={f}")
print(f"  Interphase layers: phi=[{phi_1}, {phi_2}, {phi_3}]")
print(f"  Particle radius: R={R:.6f}")

## Step 1: Compute Target Properties from 3-Layer Model

In [ ]:
K_target, G_target = compute_3layer_properties(
    kappa_inc, mu_inc, kappa_mat, mu_mat,
    phi_1, phi_2, phi_3, f
)

print("Target Effective Properties (from 3-Layer Interphase Model):")
print(f"  K_eff = {K_target:.4f}")
print(f"  G_eff = {G_target:.4f}")

## Step 2: Predict Interface Parameters using PINN

In [ ]:
# Prepare input tensor
inputs = torch.tensor([
    [K_target, G_target, kappa_inc, mu_inc, kappa_mat, mu_mat, f, R]
], dtype=torch.float32)

# Predict
with torch.no_grad():
    params = model(inputs)
    k_bar, lambda_bar, mu_bar, alpha = params[0].numpy()

print("Predicted Interface Parameters (from PINN):")
print(f"  k_bar      = {k_bar:.4f}     (normal stiffness [N/m^3])")
print(f"  lambda_bar = {lambda_bar:.6f}  (tangential Lame [N/m])")
print(f"  mu_bar     = {mu_bar:.6f}  (tangential shear [N/m])")
print(f"  alpha      = {alpha:.6f}  (interface position: 0=cohesive, 1=elastic)")

## Step 3: Verify with Extended Interface Model

In [ ]:
K_recon, G_recon = compute_effective_properties(
    kappa_inc, mu_inc, kappa_mat, mu_mat,
    k_bar, lambda_bar, mu_bar, alpha, f, R
)

# Compute errors
K_error = abs(K_recon - K_target) / K_target * 100
G_error = abs(G_recon - G_target) / G_target * 100

print("Reconstructed Effective Properties (from Extended Interface Model):")
print(f"  K_eff = {K_recon:.4f} (target: {K_target:.4f}, error: {K_error:.2f}%)")
print(f"  G_eff = {G_recon:.4f} (target: {G_target:.4f}, error: {G_error:.2f}%)")

# Pass/Fail (threshold: 5%)
passed = K_error < 5 and G_error < 5
print(f"\nResult: {'PASS' if passed else 'FAIL'} (threshold: 5%)")

## Summary Table

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"{'Property':<15} {'3-Layer Model':<18} {'Extended Interface':<18} {'Error':<10}")
print("-"*70)
print(f"{'K_eff':<15} {K_target:<18.4f} {K_recon:<18.4f} {K_error:.2f}%")
print(f"{'G_eff':<15} {G_target:<18.4f} {G_recon:<18.4f} {G_error:.2f}%")
print("-"*70)
print(f"\nInterface Parameters: k_bar={k_bar:.2f}, lambda_bar={lambda_bar:.4f}, mu_bar={mu_bar:.4f}, alpha={alpha:.4f}")

## Try Different Configurations

You can modify the parameters above to test different composite configurations!

In [ ]:
# Example: Test with stiff particles (SR = 2.0)
def test_configuration(SR, f, phi_1, phi_2, phi_3):
    """Test a specific configuration."""
    kappa_inc_test = kappa_mat * SR
    mu_inc_test = mu_mat * SR
    R_test = ((3 * f) / (4 * np.pi)) ** (1/3) * sz
    
    K_target, G_target = compute_3layer_properties(
        kappa_inc_test, mu_inc_test, kappa_mat, mu_mat,
        phi_1, phi_2, phi_3, f
    )
    
    inputs = torch.tensor([[K_target, G_target, kappa_inc_test, mu_inc_test, 
                          kappa_mat, mu_mat, f, R_test]], dtype=torch.float32)
    
    with torch.no_grad():
        params = model(inputs)
        k_bar, lambda_bar, mu_bar, alpha = params[0].numpy()
    
    K_recon, G_recon = compute_effective_properties(
        kappa_inc_test, mu_inc_test, kappa_mat, mu_mat,
        k_bar, lambda_bar, mu_bar, alpha, f, R_test
    )
    
    K_error = abs(K_recon - K_target) / K_target * 100
    G_error = abs(G_recon - G_target) / G_target * 100
    
    return K_error, G_error

# Test multiple configurations
configs = [
    (0.1, 0.3, 0.2, 0.3, 0.4, "Soft particles (SR=0.1)"),
    (0.5, 0.3, 0.2, 0.3, 0.4, "Medium particles (SR=0.5)"),
    (2.0, 0.3, 0.2, 0.3, 0.4, "Stiff particles (SR=2.0)"),
    (0.1, 0.2, 0.2, 0.3, 0.4, "Low volume fraction (f=0.2)"),
    (0.1, 0.4, 0.2, 0.3, 0.4, "High volume fraction (f=0.4)"),
]

print(f"{'Configuration':<35} {'K_error':>10} {'G_error':>10} {'Status':>8}")
print("-"*70)
for SR, f_test, p1, p2, p3, name in configs:
    K_err, G_err = test_configuration(SR, f_test, p1, p2, p3)
    status = "PASS" if K_err < 5 and G_err < 5 else "FAIL"
    print(f"{name:<35} {K_err:>9.2f}% {G_err:>9.2f}% {status:>8}")